# Taller de Minería de Datos: Detección de Vulnerabilidades con OWASP Top 10 (2021)

**Autor:** Angel Geovanny Cudco Pomagualli  
**Universidad:** UNMSM  
**Tema:** Clasificación multiclase de fragmentos de código vulnerables según OWASP Top 10 (2021)  
**Técnicas:** TF-IDF + Random Forest + Regresión Logística  
**Explicabilidad:** SHAP y LIME

---

## Objetivos

1. Construir un dataset (sintético) de fragmentos de código etiquetados según OWASP Top 10 2021.
2. Aplicar técnicas de minería de datos: EDA, preprocesamiento, vectorización TF-IDF.
3. Entrenar un modelo de clasificación multiclase para identificar el tipo de vulnerabilidad.
4. Evaluar el modelo (matriz de confusión, precision, recall, F1).
5. Explicar las decisiones del modelo usando **SHAP** (global y local) y **LIME** (texto).

## Categorías OWASP Top 10 (2021)

| Código | Categoría |
|--------|-----------|
| A01 | Broken Access Control |
| A02 | Cryptographic Failures |
| A03 | Injection (SQLi, XSS, Command Injection) |
| A04 | Insecure Design |
| A05 | Security Misconfiguration |
| A06 | Vulnerable and Outdated Components |
| A07 | Identification and Authentication Failures |
| A08 | Software and Data Integrity Failures |
| A09 | Security Logging and Monitoring Failures |
| A10 | Server-Side Request Forgery (SSRF) |

> Fuente oficial: [OWASP Top 10 2021](https://owasp.org/Top10/)


## 1. Instalación de dependencias

Si trabajas en Google Colab o un entorno limpio, ejecuta la siguiente celda. Si ya tienes las librerías instaladas, puedes omitirla.

In [ ]:
# !pip install -q scikit-learn pandas numpy matplotlib seaborn shap lime wordcloud

## 2. Importación de librerías

In [ ]:
import re
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, ConfusionMatrixDisplay)

import shap
from lime.lime_text import LimeTextExplainer

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print('Librerías cargadas correctamente.')

## 3. Construcción del dataset sintético

Generamos fragmentos de código en distintos lenguajes (Python, PHP, JavaScript, Java, SQL) representativos de cada categoría OWASP. En un escenario real podrías reemplazar este paso por un dataset de Kaggle como:

- [DiverseVul](https://www.kaggle.com/datasets/) (vulnerabilidades CWE)
- [Devign Dataset](https://sites.google.com/view/devign)
- [SARD - NIST Software Assurance Reference Dataset](https://samate.nist.gov/SARD/)

Para fines didácticos, creamos plantillas que producen ~80-100 fragmentos por categoría.

In [ ]:
# Plantillas de fragmentos de código por categoría OWASP
PLANTILLAS = {
    "A01_Broken_Access_Control": [
        "@app.route('/admin/{path}')\ndef admin_panel():\n    return render_template('admin.html')  # sin verificar rol",
        "def get_user_profile(user_id):\n    return db.query(f'SELECT * FROM users WHERE id={user_id}')  # IDOR",
        "if request.method == 'POST':\n    user.is_admin = request.form['is_admin']  # asignación masiva",
        "<?php\n$file = $_GET['file'];\nreadfile('/var/www/uploads/' . $file); ?>  // path traversal",
        "app.get('/api/orders/:id', (req, res) => {{\n  Order.findById(req.params.id).then(o => res.json(o)); // sin auth\n}});",
        "def delete_account(account_id):\n    Account.objects.get(id=account_id).delete()  # falta check de owner",
        "function viewDocument(docId) {{\n  return fetch('/docs/' + docId);  // referencia directa insegura\n}}",
        "@PreAuthorize(\"permitAll()\")\npublic User getUser(Long id) {{ return repo.findById(id); }}",
        "if user.role == 'guest':\n    pass  # se permite acceso sin restricción a {path}",
        "location /admin/ {{ allow all; }}  # nginx config sin restricción de IP",
    ],
    "A02_Cryptographic_Failures": [
        "import hashlib\npassword_hash = hashlib.md5(password.encode()).hexdigest()  # MD5 obsoleto",
        "cipher = DES.new(key, DES.MODE_ECB)  # DES con ECB es inseguro",
        "ssl_context = ssl.SSLContext(ssl.PROTOCOL_TLSv1)  # TLS 1.0 obsoleto",
        "api_key = 'sk_live_{path}1234567890abcdef'  # secreto hardcodeado",
        "<?php $hash = sha1($_POST['password']); ?>  // SHA1 inseguro",
        "Cipher c = Cipher.getInstance(\"AES/ECB/PKCS5Padding\"); // modo ECB",
        "fetch('http://api.example.com/login', {{method:'POST', body: pwd}})  // HTTP plano",
        "key = b'1234567890123456'  # clave estática reutilizada",
        "openssl_encrypt($data, 'rc4', $key)  // RC4 está roto",
        "self.token = base64.b64encode(password.encode())  # base64 NO es cifrado {path}",
    ],
    "A03_Injection": [
        "query = \"SELECT * FROM users WHERE name='\" + username + \"'\"  # SQL injection",
        "cursor.execute(f'SELECT * FROM products WHERE id={product_id}')  # SQLi por f-string",
        "os.system('ping ' + user_input)  # command injection",
        "<?php $q = \"SELECT * FROM u WHERE n='\".$_GET['n'].\"'\"; ?>  // SQLi PHP",
        "document.getElementById('out').innerHTML = userInput;  // XSS DOM",
        "<div>{{ user_comment|safe }}</div>  <!-- XSS reflejado en Jinja -->",
        "Runtime.getRuntime().exec(\"sh -c \" + cmd);  // command injection Java",
        "eval(request.GET.get('expr'))  # eval con input del usuario",
        "subprocess.call('cat ' + filename, shell=True)  # shell=True peligroso {path}",
        "ldap_search = '(uid=' + user + ')'  # LDAP injection",
    ],
    "A04_Insecure_Design": [
        "def reset_password(email):\n    new_pwd = '123456'  # contraseña predecible al resetear",
        "if user.age < 18:\n    pass  # no se valida flujo de menores en {path}",
        "def transfer_funds(amount, to):\n    # sin límite ni verificación de doble factor\n    do_transfer(amount, to)",
        "<?php // CAPTCHA omitido en login para 'mejorar UX' ?>",
        "def signup(email):\n    # no se valida si el correo ya existe ni dominio corporativo\n    User.create(email)",
        "// flujo de checkout permite precio negativo: total = price * qty;",
        "def grant_discount(code):\n    # cupones reutilizables sin expiración\n    apply(code)",
        "if request.path.startswith('/api'):\n    skip_csrf = True  # decisión de diseño riesgosa",
        "# lógica de negocio confía ciegamente en el cliente para calcular total {path}",
        "def upload(file):\n    save(file)  # no se define tamaño máximo ni tipo permitido",
    ],
    "A05_Security_Misconfiguration": [
        "DEBUG = True  # en producción expone stacktraces",
        "app.config['SECRET_KEY'] = 'changeme'  # clave por defecto",
        "<Directory />\n  Options Indexes FollowSymLinks\n  AllowOverride None\n</Directory>",
        "CORS(app, resources={{r'/*': {{'origins': '*'}}}})  # CORS abierto",
        "server {{ listen 80; autoindex on; }}  # listado de directorios habilitado",
        "spring.h2.console.enabled=true  # consola H2 expuesta en prod",
        "X-Powered-By: PHP/5.6.0  // header revela versión obsoleta",
        "docker run --privileged -p 27017:27017 mongo  # Mongo sin auth expuesto {path}",
        "app.run(host='0.0.0.0', debug=True)  # debug en producción",
        "headers = {{}}  # faltan X-Frame-Options, CSP, HSTS",
    ],
    "A06_Vulnerable_Components": [
        "# requirements.txt\nDjango==1.11.0  # versión con CVEs conocidos",
        "<script src='https://code.jquery.com/jquery-1.4.2.min.js'></script>  <!-- jQuery vulnerable -->",
        "log4j-core 2.14.1  # vulnerable a Log4Shell CVE-2021-44228",
        "openssl 1.0.1f  # Heartbleed CVE-2014-0160",
        "\"lodash\": \"4.17.4\"  // CVE prototype pollution",
        "FROM node:8  # imagen base sin soporte",
        "spring-core 4.3.0 // vulnerable a Spring4Shell {path}",
        "pip install flask==0.10.0  # versión antigua sin parches",
        "<dependency>struts2-core 2.3.15</dependency>  <!-- CVE-2017-5638 -->",
        "composer.json: \"symfony/http-kernel\": \"2.7.0\"  // múltiples CVEs",
    ],
    "A07_Auth_Failures": [
        "if password == stored_password:  # comparación sin hash ni constant_time",
        "session['user'] = username  # sin regenerar ID tras login -> session fixation",
        "def login(u, p):\n    if u in users and users[u] == p: return True  # sin rate limiting {path}",
        "<?php setcookie('session', $sid); ?>  // sin HttpOnly ni Secure",
        "if attempts < 9999:\n    try_login()  # sin bloqueo por intentos fallidos",
        "JWT.create().sign(Algorithm.none());  // JWT sin firma",
        "password_policy = r'.*'  # política de contraseñas inexistente",
        "def remember_me(token):\n    return base64.b64decode(token)  # token predecible",
        "# 2FA opcional incluso para administradores",
        "auth_token = uuid.uuid1()  # UUID1 es predecible (timestamp)",
    ],
    "A08_Integrity_Failures": [
        "pickle.loads(request.data)  # deserialización insegura",
        "yaml.load(user_input)  # sin SafeLoader, ejecuta objetos",
        "<script src='http://cdn.untrusted.com/lib.js'></script>  <!-- sin SRI -->",
        "ObjectInputStream in = new ObjectInputStream(socket.getInputStream()); // Java deser",
        "auto_update_url = 'http://updates.app.com'  # actualizaciones sin firma {path}",
        "npm install --no-verify-signatures pkg  # paquetes sin verificación",
        "exec(downloaded_code)  # ejecuta código bajado sin validar checksum",
        "git clone http://github.com/repo.git  # clone sin verificar firma GPG",
        "Marshal.load(params[:data])  # Ruby deserialización",
        "<link rel='stylesheet' href='https://x.com/a.css'>  <!-- sin integrity hash -->",
    ],
    "A09_Logging_Failures": [
        "try:\n    do_payment()\nexcept:\n    pass  # excepción silenciada, sin log",
        "logger.info(f'Usuario {{user}} ingresó password={{pwd}}')  # PII en logs",
        "# no se registran intentos fallidos de login {path}",
        "logging.basicConfig(level=logging.CRITICAL)  # casi nada se loguea",
        "<?php error_reporting(0); ?>  // errores ocultados sin log alterno",
        "catch (Exception e) {{}}  // Java: se traga la excepción",
        "if not user.is_admin: return  # sin auditoría de accesos denegados",
        "# eventos de seguridad solo se guardan localmente y se rotan a las 24h",
        "syslog_server = None  # sin envío centralizado de logs",
        "console.log(creditCard);  // datos sensibles en consola del navegador",
    ],
    "A10_SSRF": [
        "url = request.GET.get('url')\nresp = requests.get(url)  # SSRF clásico",
        "img = urllib.request.urlopen(user_supplied_url)  # SSRF en fetch de imagen",
        "<?php $data = file_get_contents($_GET['endpoint']); ?>  // SSRF PHP",
        "fetch(req.query.target).then(r => r.text())  // SSRF Node.js {path}",
        "curl_setopt($ch, CURLOPT_URL, $_POST['callback']);  // SSRF curl",
        "webhook = WebhookClient(url=request.json['hook'])  # sin allowlist de dominios",
        "response = httpx.get(f'http://internal.svc/{{user_path}}')  # path injection -> SSRF",
        "preview = requests.get(link, allow_redirects=True)  # redirección puede ir a 169.254.169.254",
        "ImageIO.read(new URL(userUrl));  // Java SSRF",
        "axios.get(req.body.profileUrl)  // SSRF a metadata cloud",
    ],
}

# Generar variantes sintéticas para enriquecer el dataset
def generar_dataset(plantillas, n_por_clase=80, seed=SEED):
    random.seed(seed)
    filas = []
    nombres_var = ['user', 'admin', 'data', 'token', 'param', 'value', 'item', 'config', 'session', 'request']
    for etiqueta, ejemplos in plantillas.items():
        for _ in range(n_por_clase):
            base = random.choice(ejemplos)
            # pequeñas mutaciones para evitar duplicados
            var = random.choice(nombres_var) + str(random.randint(1, 999))
            fragmento = base.replace('{path}', var)
            # ruido: agregar comentario aleatorio
            if random.random() < 0.3:
                fragmento += f"\n# nota: revisar {var}"
            filas.append({'codigo': fragmento, 'etiqueta': etiqueta})
    df = pd.DataFrame(filas).sample(frac=1, random_state=seed).reset_index(drop=True)
    return df

df = generar_dataset(PLANTILLAS, n_por_clase=80)
print(f'Dataset generado: {len(df)} fragmentos, {df.etiqueta.nunique()} clases')
df.head(5)

## 4. Análisis Exploratorio de Datos (EDA)

Revisamos la distribución de clases, la longitud de los fragmentos y vocabulario característico de cada categoría.

In [ ]:
# Distribución de clases
plt.figure(figsize=(11, 5))
order = sorted(df['etiqueta'].unique())
sns.countplot(data=df, y='etiqueta', order=order, palette='viridis')
plt.title('Distribución de fragmentos por categoría OWASP Top 10 (2021)')
plt.xlabel('Cantidad de fragmentos')
plt.ylabel('Categoría')
plt.tight_layout()
plt.show()

In [ ]:
# Longitud (caracteres y tokens) por categoría
df['n_caracteres'] = df['codigo'].str.len()
df['n_tokens'] = df['codigo'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.boxplot(data=df, x='n_caracteres', y='etiqueta', order=order, ax=axes[0], palette='magma')
axes[0].set_title('Longitud (caracteres)')
sns.boxplot(data=df, x='n_tokens', y='etiqueta', order=order, ax=axes[1], palette='magma')
axes[1].set_title('Longitud (tokens)')
plt.tight_layout()
plt.show()

df[['n_caracteres', 'n_tokens']].describe().round(2)

In [ ]:
# Palabras más frecuentes por categoría (top 8)
from collections import Counter

def tokenizar_codigo(texto):
    # tokens alfanuméricos + algunos símbolos clave del código
    return re.findall(r"[A-Za-z_][A-Za-z0-9_]+", texto.lower())

top_palabras = {}
for clase in order:
    tokens = []
    for t in df.loc[df['etiqueta'] == clase, 'codigo']:
        tokens.extend(tokenizar_codigo(t))
    top_palabras[clase] = Counter(tokens).most_common(8)

for clase, palabras in top_palabras.items():
    print(f'\n>>> {clase}')
    print('   ', ', '.join([f'{w}({c})' for w, c in palabras]))

## 5. Preprocesamiento y vectorización

Convertimos los fragmentos de código a vectores numéricos usando **TF-IDF** con n-gramas (1, 2). Mantenemos símbolos comunes del código mediante un tokenizador personalizado.

In [ ]:
def tokenizer_codigo(texto):
    # Captura identificadores, números y símbolos comunes
    texto = texto.lower()
    return re.findall(r"[a-z_][a-z0-9_]*|[0-9]+|[=<>!+\-*/%&|^]+", texto)

X = df['codigo']
y = df['etiqueta']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)

vectorizer = TfidfVectorizer(
    tokenizer=tokenizer_codigo,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True,
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f'Vocabulario: {len(vectorizer.vocabulary_)} términos')
print(f'X_train: {X_train_vec.shape}  |  X_test: {X_test_vec.shape}')

## 6. Modelado: clasificación multiclase

Entrenamos dos modelos para comparar:

1. **Regresión Logística** (lineal, interpretable, buen baseline para texto).
2. **Random Forest** (no lineal, robusto, base para SHAP TreeExplainer).

In [ ]:
# Modelo 1: Regresión Logística
log_reg = LogisticRegression(
    max_iter=1000, C=2.0, solver='lbfgs', random_state=SEED
)
log_reg.fit(X_train_vec, y_train)
y_pred_lr = log_reg.predict(X_test_vec)

print('=== Regresión Logística ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}\n')
print(classification_report(y_test, y_pred_lr, zero_division=0))

In [ ]:
# Modelo 2: Random Forest
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=-1,
)
rf.fit(X_train_vec, y_train)
y_pred_rf = rf.predict(X_test_vec)

print('=== Random Forest ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}\n')
print(classification_report(y_test, y_pred_rf, zero_division=0))

## 7. Evaluación visual: matriz de confusión

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
clases = sorted(y.unique())

for ax, (nombre, y_pred) in zip(axes, [('Regresión Logística', y_pred_lr), ('Random Forest', y_pred_rf)]):
    cm = confusion_matrix(y_test, y_pred, labels=clases)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=[c[:6] for c in clases],
                yticklabels=[c[:6] for c in clases], ax=ax, cbar=False)
    ax.set_title(f'Matriz de Confusión — {nombre}')
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')

plt.tight_layout()
plt.show()

In [ ]:
# Elegimos el mejor modelo en base a accuracy
acc_lr = accuracy_score(y_test, y_pred_lr)
acc_rf = accuracy_score(y_test, y_pred_rf)
mejor_modelo = rf if acc_rf >= acc_lr else log_reg
nombre_mejor = 'Random Forest' if acc_rf >= acc_lr else 'Regresión Logística'
print(f'Mejor modelo: {nombre_mejor} (accuracy = {max(acc_lr, acc_rf):.4f})')

## 8. Explicabilidad global y local con SHAP

SHAP (SHapley Additive exPlanations) descompone la predicción del modelo en aportes por feature. Usamos **TreeExplainer** sobre el Random Forest (rápido y exacto para árboles).

In [ ]:
# Para SHAP necesitamos features densas y nombres
feature_names = np.array(vectorizer.get_feature_names_out())
X_test_dense = X_test_vec.toarray()

# Usamos una muestra para acelerar el cálculo
n_muestra = min(50, X_test_dense.shape[0])
idx_muestra = np.random.RandomState(SEED).choice(X_test_dense.shape[0], n_muestra, replace=False)
X_sample = X_test_dense[idx_muestra]

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_sample, check_additivity=False)
print(f'SHAP listo. Tipo: {type(shap_values)}. Clases: {len(rf.classes_)}')

In [ ]:
# Importancia global: top features promediadas sobre todas las clases
# shap_values es lista de arrays (n_clases) cada uno (n_muestra, n_features)
import numpy as _np

# Compatibilidad con distintas versiones de SHAP
if isinstance(shap_values, list):
    abs_vals = _np.mean([_np.abs(sv) for sv in shap_values], axis=0)  # (n_muestra, n_features)
else:
    # shap >= 0.45 puede devolver array (n_muestra, n_features, n_clases)
    abs_vals = _np.mean(_np.abs(shap_values), axis=-1)

mean_importance = abs_vals.mean(axis=0)
top_idx = _np.argsort(mean_importance)[-20:][::-1]

plt.figure(figsize=(10, 7))
sns.barplot(x=mean_importance[top_idx], y=feature_names[top_idx], palette='rocket')
plt.title('Top 20 features (tokens / n-gramas) más influyentes — SHAP global')
plt.xlabel('Importancia media |SHAP|')
plt.tight_layout()
plt.show()

In [ ]:
# Explicación local: tomamos un fragmento del test y vemos por qué se clasificó así
ejemplo_idx = idx_muestra[0]
texto_ejemplo = X_test.iloc[ejemplo_idx]
clase_real = y_test.iloc[ejemplo_idx]
clase_pred = rf.predict(X_test_vec[ejemplo_idx])[0]

print('--- FRAGMENTO ANALIZADO ---')
print(texto_ejemplo)
print(f'\nClase real: {clase_real}')
print(f'Clase predicha: {clase_pred}')

# Índice de la clase predicha
idx_clase = list(rf.classes_).index(clase_pred)

# Obtener vector SHAP del ejemplo para la clase predicha
if isinstance(shap_values, list):
    shap_ej = shap_values[idx_clase][0]
else:
    shap_ej = shap_values[0, :, idx_clase]

# Top tokens que empujaron hacia la clase predicha
contrib = pd.DataFrame({'feature': feature_names, 'shap': shap_ej})
contrib['abs'] = contrib['shap'].abs()
top_contrib = contrib.sort_values('abs', ascending=False).head(15)

plt.figure(figsize=(10, 6))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in top_contrib['shap']]
plt.barh(top_contrib['feature'][::-1], top_contrib['shap'][::-1], color=colors[::-1])
plt.axvline(0, color='black', linewidth=0.6)
plt.title(f'Aportes SHAP locales para clase predicha: {clase_pred}')
plt.xlabel('Valor SHAP (verde: empuja hacia la clase, rojo: en contra)')
plt.tight_layout()
plt.show()

## 9. Explicabilidad textual con LIME

LIME (Local Interpretable Model-agnostic Explanations) perturba el texto original y aprende un modelo lineal local para explicar la predicción en términos de **palabras del código**, lo cual es muy intuitivo cuando trabajamos con fragmentos de programación.

In [ ]:
# Pipeline (vectorizer + modelo) para que LIME reciba texto crudo
pipeline = Pipeline([('tfidf', vectorizer), ('clf', rf)])

# Verificación rápida
sample_text = X_test.iloc[0]
probs = pipeline.predict_proba([sample_text])[0]
top3 = pd.Series(probs, index=rf.classes_).sort_values(ascending=False).head(3)
print('Top 3 clases más probables para el primer fragmento del test:')
print(top3.round(3))

In [ ]:
explainer_lime = LimeTextExplainer(class_names=list(rf.classes_))

# Tomamos algunos fragmentos del test para explicar
indices_lime = [0, 1, 2]

for i in indices_lime:
    texto = X_test.iloc[i]
    real = y_test.iloc[i]
    pred = pipeline.predict([texto])[0]
    idx_pred = list(rf.classes_).index(pred)

    exp = explainer_lime.explain_instance(
        texto,
        pipeline.predict_proba,
        num_features=10,
        labels=[idx_pred],
    )

    print('='*80)
    print(f'Fragmento #{i}')
    print(texto)
    print(f'  Real: {real}  |  Predicho: {pred}')
    print('  Palabras más influyentes (peso LIME):')
    for palabra, peso in exp.as_list(label=idx_pred):
        signo = '+' if peso > 0 else '-'
        print(f'   {signo} {palabra:25s}  {peso:+.4f}')

In [ ]:
# Visualización rica de LIME (HTML embebido) para el primer fragmento
from IPython.display import HTML, display

texto = X_test.iloc[0]
pred = pipeline.predict([texto])[0]
idx_pred = list(rf.classes_).index(pred)
exp = explainer_lime.explain_instance(texto, pipeline.predict_proba,
                                      num_features=10, labels=[idx_pred])

# Renderizamos el HTML manualmente (compatible con IPython moderno)
html = exp.as_html(labels=[idx_pred])
display(HTML(html))

## 10. Inferencia: clasifica tus propios fragmentos de código

Definimos una función `analizar_codigo(...)` que recibe un fragmento de código y devuelve:

1. La **categoría OWASP predicha** y sus probabilidades.
2. La explicación **LIME** (palabras del código que más influyeron).
3. La explicación **SHAP** (aportes de cada token a la clase predicha).

Después mostramos varios **ejemplos predefinidos** (que el modelo nunca vio en entrenamiento) y una **celda donde puedes pegar tu propio código**.

In [ ]:
# Función principal de inferencia + explicabilidad
from IPython.display import HTML, display, Markdown

DESCRIPCIONES_OWASP = {
    'A01_Broken_Access_Control': 'Control de acceso roto: el usuario puede actuar fuera de sus permisos (IDOR, path traversal, escalada de privilegios).',
    'A02_Cryptographic_Failures': 'Fallos criptográficos: uso de algoritmos débiles (MD5, SHA1, DES, RC4), claves hardcodeadas, TLS obsoleto.',
    'A03_Injection': 'Inyección: SQLi, XSS, command injection, LDAP injection. El input del usuario llega a un intérprete sin sanitizar.',
    'A04_Insecure_Design': 'Diseño inseguro: falla lógica del negocio (contraseñas predecibles, falta de límites, flujos sin validar).',
    'A05_Security_Misconfiguration': 'Mala configuración: DEBUG=True en prod, CORS abierto, claves por defecto, listado de directorios.',
    'A06_Vulnerable_Components': 'Componentes vulnerables: librerías/frameworks con CVEs conocidos (Log4Shell, Heartbleed, etc.).',
    'A07_Auth_Failures': 'Fallos de autenticación: sin rate limiting, tokens predecibles, JWT sin firma, contraseñas sin hash.',
    'A08_Integrity_Failures': 'Fallos de integridad: deserialización insegura (pickle, yaml.load), scripts sin SRI, updates sin firma.',
    'A09_Logging_Failures': 'Fallos de logging/monitoreo: excepciones silenciadas, PII en logs, sin auditoría de eventos de seguridad.',
    'A10_SSRF': 'Server-Side Request Forgery: el servidor hace requests a URLs controladas por el usuario sin allowlist.',
}

def analizar_codigo(codigo, top_k=3, num_features_lime=10, mostrar_shap=True, mostrar_lime=True):
    """Clasifica un fragmento de código y lo explica con LIME y SHAP."""
    # --- 1. Predicción ---
    probas = pipeline.predict_proba([codigo])[0]
    clases = list(rf.classes_)
    ranking = pd.Series(probas, index=clases).sort_values(ascending=False)
    pred = ranking.index[0]
    idx_pred = clases.index(pred)
    descripcion = DESCRIPCIONES_OWASP.get(pred, '')

    # --- 2. Encabezado ---
    display(Markdown("### Resultado de la clasificación"))
    display(Markdown(f"**Código analizado:**\n\n```\n{codigo.strip()}\n```"))
    display(Markdown(
        f"**Categoría predicha:** `{pred}`  \n"
        f"**Confianza:** `{probas[idx_pred]*100:.2f}%`\n\n"
        f"_{descripcion}_"
    ))

    # --- 3. Top-K probabilidades ---
    display(Markdown(f"**Top {top_k} categorías más probables:**"))
    top_df = ranking.head(top_k).to_frame('probabilidad')
    top_df['probabilidad'] = (top_df['probabilidad']*100).round(2).astype(str) + ' %'
    display(top_df)

    # Gráfico de barras
    plt.figure(figsize=(9, 4))
    colores = ['#27ae60' if c == pred else '#95a5a6' for c in ranking.index]
    plt.barh(ranking.index[::-1], ranking.values[::-1]*100, color=colores[::-1])
    plt.xlabel('Probabilidad (%)')
    plt.title('Distribución de probabilidades por categoría OWASP')
    plt.tight_layout()
    plt.show()

    # --- 4. Explicación LIME ---
    if mostrar_lime:
        display(Markdown('### Explicación LIME (palabras del código más influyentes)'))
        exp = explainer_lime.explain_instance(
            codigo, pipeline.predict_proba,
            num_features=num_features_lime, labels=[idx_pred]
        )
        lime_df = pd.DataFrame(exp.as_list(label=idx_pred), columns=['palabra', 'peso_LIME'])
        lime_df['efecto'] = lime_df['peso_LIME'].apply(
            lambda v: f'empuja hacia {pred}' if v > 0 else f'aleja de {pred}'
        )
        display(lime_df)
        display(HTML(exp.as_html(labels=[idx_pred])))

    # --- 5. Explicación SHAP ---
    if mostrar_shap:
        display(Markdown('### Explicación SHAP (tokens / n-gramas TF-IDF)'))
        vec = vectorizer.transform([codigo]).toarray()
        sv = explainer.shap_values(vec, check_additivity=False)
        if isinstance(sv, list):
            shap_vec = sv[idx_pred][0]
        else:
            shap_vec = sv[0, :, idx_pred]
        contrib = pd.DataFrame({'feature': feature_names, 'shap': shap_vec})
        contrib['abs'] = contrib['shap'].abs()
        top = contrib.sort_values('abs', ascending=False).head(12)
        top = top[top['abs'] > 0]
        if len(top) == 0:
            display(Markdown('_No se encontraron tokens del fragmento en el vocabulario TF-IDF._'))
        else:
            plt.figure(figsize=(9, 5))
            colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in top['shap']]
            plt.barh(top['feature'][::-1], top['shap'][::-1], color=colors[::-1])
            plt.axvline(0, color='black', linewidth=0.6)
            plt.title(f'Aportes SHAP para la clase predicha: {pred}')
            plt.xlabel('Valor SHAP (verde: a favor, rojo: en contra)')
            plt.tight_layout()
            plt.show()

    return {'prediccion': pred, 'confianza': float(probas[idx_pred]), 'ranking': ranking}

print('Función analizar_codigo() lista para usar.')

### 10.1 Ejemplos predefinidos

Probamos la función con fragmentos de código vulnerable **que el modelo nunca vio durante el entrenamiento**.

In [ ]:
# Ejemplo 1: SQL Injection (debería predecir A03 Injection)
codigo_1 = """
def buscar_usuario(nombre):
    conn = sqlite3.connect('app.db')
    cur = conn.cursor()
    query = "SELECT * FROM clientes WHERE nombre = '" + nombre + "'"
    cur.execute(query)
    return cur.fetchall()
"""
_ = analizar_codigo(codigo_1)

In [ ]:
# Ejemplo 2: Hash MD5 (debería predecir A02 Cryptographic Failures)
codigo_2 = """
import hashlib
def guardar_password(usuario, password):
    h = hashlib.md5(password.encode()).hexdigest()
    db.execute("INSERT INTO users VALUES (?, ?)", (usuario, h))
"""
_ = analizar_codigo(codigo_2)

In [ ]:
# Ejemplo 3: SSRF (debería predecir A10)
codigo_3 = """
import requests
@app.route('/fetch')
def fetch_url():
    target = request.args.get('url')
    r = requests.get(target, timeout=5)
    return r.text
"""
_ = analizar_codigo(codigo_3)

In [ ]:
# Ejemplo 4: Deserialización insegura (debería predecir A08)
codigo_4 = """
import pickle
def cargar_sesion(data_bytes):
    sesion = pickle.loads(data_bytes)
    return sesion
"""
_ = analizar_codigo(codigo_4)

In [ ]:
# Ejemplo 5: DEBUG=True en producción (debería predecir A05)
codigo_5 = """
app = Flask(__name__)
app.config['DEBUG'] = True
app.config['SECRET_KEY'] = 'changeme'
if __name__ == '__main__':
    app.run(host='0.0.0.0', debug=True)
"""
_ = analizar_codigo(codigo_5)

### 10.2 Tu propio fragmento de código

Edita la variable `mi_codigo` con el código que quieras analizar y ejecuta la celda. Puedes pegar código en cualquier lenguaje (Python, PHP, JS, Java, SQL...).

In [ ]:
# >>> EDITA AQUÍ TU FRAGMENTO DE CÓDIGO <<<
mi_codigo = """
# Ejemplo por defecto: XSS reflejado en una vista Flask
from flask import request
@app.route('/saludo')
def saludo():
    nombre = request.args.get('nombre')
    return "<h1>Hola " + nombre + "</h1>"
"""

resultado = analizar_codigo(mi_codigo, top_k=3, num_features_lime=10)
print(f"\n>>> Predicción final: {resultado['prediccion']} ({resultado['confianza']*100:.1f}%)")

### 10.3 (Opcional) Widget interactivo

Si tu entorno soporta `ipywidgets` (Jupyter Lab/Notebook clásico, Google Colab), tendrás un cuadro de texto y un botón para analizar código sin editar celdas.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    txt = widgets.Textarea(
        value="def login(u, p):\n    if u in users and users[u] == p:\n        return True",
        placeholder='Pega aquí tu fragmento de código...',
        description='Código:',
        layout=widgets.Layout(width='100%', height='180px')
    )
    btn = widgets.Button(description='Analizar', button_style='success', icon='search')
    out = widgets.Output()

    def on_click(_):
        with out:
            clear_output()
            analizar_codigo(txt.value)

    btn.on_click(on_click)
    display(txt, btn, out)
except ImportError:
    print("ipywidgets no está disponible. Usa la celda 10.2 (variable mi_codigo) en su lugar.")

## 11. Conclusiones y trabajo futuro

### Hallazgos
- Es posible clasificar fragmentos de código en categorías OWASP Top 10 (2021) usando representaciones simples como TF-IDF + clasificadores clásicos.
- **Random Forest** suele superar a Regresión Logística cuando hay interacciones entre tokens (p. ej. `pickle` + `loads` para A08).
- **SHAP** muestra qué tokens son globalmente más importantes (p. ej. `md5`, `pickle.loads`, `eval`, `system`, `requests.get`).
- **LIME** facilita explicar a un desarrollador *qué palabras del fragmento* dispararon la alerta, útil para reportes de seguridad.

### Limitaciones
- El dataset es **sintético**: el modelo aprende patrones de las plantillas. En producción, se debe entrenar con datasets reales (DiverseVul, Devign, SARD, CVE-fixes).
- No se considera el contexto del repositorio ni el control de flujo (AST, grafos de dependencias).
- Las clases pueden solaparse (p. ej. una inyección SQL también puede ser fallo de diseño).

### Próximos pasos
1. Reemplazar el dataset sintético por un dataset real de Kaggle / GitHub.
2. Usar embeddings de código (CodeBERT, GraphCodeBERT) en lugar de TF-IDF.
3. Evaluar con métricas por familia CWE además de OWASP.
4. Integrar el modelo como un pre-commit hook o un plugin para CI/CD.

---

**Referencias:**
- [OWASP Top 10 2021](https://owasp.org/Top10/)
- [SHAP — Lundberg & Lee, 2017](https://github.com/shap/shap)
- [LIME — Ribeiro et al., 2016](https://github.com/marcotcr/lime)
